# WoW Economy Forecaster -- Backtest and Evaluation

This notebook presents the formal evaluation of the forecasting pipeline. A model without rigorous backtesting is just a random number generator with extra steps.

We evaluate along four axes:

1. **Point accuracy**: MAE, RMSE, MAPE -- how close are predictions to actuals?
2. **Directional accuracy**: Did we predict the right direction of price movement? (Critical for buy/sell decisions.)
3. **Calibration**: Are our confidence intervals well-calibrated? If we say 80% CI, do ~80% of actuals fall within it?
4. **Recommendation quality**: Does the scoring formula produce sensible, explainable recommendations?

## Walk-Forward Methodology

Time-series evaluation must never use random train/test splits (information leakage). Instead, we use **walk-forward validation**:

```
Fold 1:  [======= TRAIN ========] [TEST]_______________
Fold 2:  [========= TRAIN =========] [TEST]____________
Fold 3:  [=========== TRAIN ===========] [TEST]________
Fold 4:  [============= TRAIN =============] [TEST]____
          t0                                          now
```

Each fold:
1. Trains on all data up to `train_end`.
2. Predicts `horizon_days` forward from `train_end`.
3. Compares predictions to actual prices on the test date.
4. Advances the window and repeats.

This mimics real deployment exactly: the model never sees future data during training. The expanding window (rather than sliding) reflects the actual training strategy where we always use all available history.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "wow_forecaster.db"
BACKTEST_DIR = PROJECT_ROOT / "data" / "processed" / "backtest"
RECS_DIR = PROJECT_ROOT / "data" / "outputs" / "recommendations"
ARTIFACT_DIR = PROJECT_ROOT / "data" / "outputs" / "model_artifacts"
REALM = "us"

print(f"Project root   : {PROJECT_ROOT}")
print(f"Backtest dir   : {BACKTEST_DIR}  (exists: {BACKTEST_DIR.exists()})")
print(f"Recommendations: {RECS_DIR}  (exists: {RECS_DIR.exists()})")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────

import sqlite3
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from wow_forecaster.viz.theme import (
    ACTION_COLORS,
    CATEGORY_COLORS,
    CI_QUALITY_COLORS,
    HORIZON_COLORS,
    RISK_COLORS,
    WOW_PALETTE,
    apply_wow_theme,
)
from wow_forecaster.viz.data_queries import (
    fetch_archetypes,
    fetch_backtest_predictions,
    fetch_backtest_summary,
    fetch_forecast_data,
    fetch_recommendation_scores,
)
from wow_forecaster.viz.charts.backtest_chart import (
    plot_actual_vs_predicted_scatter,
    plot_directional_accuracy_heatmap,
    plot_metrics_by_category,
    plot_residual_distribution,
    plot_walk_forward_mae,
)
from wow_forecaster.viz.charts.recommendation_chart import (
    plot_action_distribution,
    plot_roi_vs_uncertainty,
    plot_score_tornado,
)

warnings.filterwarnings("ignore", category=FutureWarning)

fig_tmp, _ = plt.subplots()
apply_wow_theme(fig_tmp)
plt.close(fig_tmp)

print("Imports OK.")

---
## 1. Overall Metrics

We start with aggregate metrics across all folds and archetypes. These tell us the model's average performance.

### Metric interpretation guide

| Metric | What it measures | Good value |
|--------|-----------------|------------|
| **MAE** | Average gold error | Lower is better; context-dependent on price scale |
| **RMSE** | Gold error with outlier penalty | RMSE >> MAE signals occasional catastrophic misses |
| **MAPE** | Percentage error (cross-category comparable) | < 10% is strong; < 20% is acceptable |
| **Directional Accuracy** | Fraction of correct up/down predictions | > 55% is meaningful signal; 50% = random |

MAPE is computed with a floor (`MAPE_EPSILON = 0.01g`) to prevent near-zero prices from creating infinite percentage errors.

In [ ]:
# ── Load backtest results ────────────────────────────────────────────────────

df_predictions = fetch_backtest_predictions(BACKTEST_DIR, REALM)
df_summary = fetch_backtest_summary(BACKTEST_DIR, REALM)

if not df_predictions.empty:
    print(f"Backtest predictions loaded: {len(df_predictions):,}")
    print(f"Columns: {df_predictions.columns.tolist()}")
    if "horizon" in df_predictions.columns or "horizon_days" in df_predictions.columns:
        h_col = "horizon" if "horizon" in df_predictions.columns else "horizon_days"
        print(f"Horizons: {sorted(df_predictions[h_col].unique())}")
else:
    print(
        "No backtest predictions found.\n"
        "Run the backtest pipeline to generate per-prediction CSVs.\n"
        f"Expected location: {BACKTEST_DIR}"
    )

In [ ]:
# ── Backtest summary table ───────────────────────────────────────────────────

if not df_summary.empty:
    print("Backtest Summary Metrics:")
    print("=" * 60)
    display_cols = [c for c in ["model_name", "horizon", "mae", "rmse", "mape",
                                 "directional_accuracy", "n_evaluated"]
                    if c in df_summary.columns]
    print(df_summary[display_cols].to_string(index=False))
else:
    print("No backtest summary data available.")

In [ ]:
# ── Compute aggregate metrics from per-prediction data ───────────────────────

if not df_predictions.empty and "actual_price" in df_predictions.columns:
    # Filter to rows with both actual and predicted
    valid = df_predictions[
        df_predictions["actual_price"].notna() &
        df_predictions["predicted_price"].notna()
    ].copy()
    valid["actual_price"] = valid["actual_price"].astype(float)
    valid["predicted_price"] = valid["predicted_price"].astype(float)

    if not valid.empty:
        errors = valid["actual_price"] - valid["predicted_price"]
        abs_errors = errors.abs()

        mae = abs_errors.mean()
        rmse = np.sqrt((errors ** 2).mean())
        mape_mask = valid["actual_price"] >= 0.01
        mape = (abs_errors[mape_mask] / valid["actual_price"][mape_mask]).mean()

        print(f"Aggregate metrics ({len(valid):,} predictions):")
        print(f"  MAE   : {mae:.2f} gold")
        print(f"  RMSE  : {rmse:.2f} gold")
        print(f"  MAPE  : {mape:.2%}")

        if "last_known_price" in valid.columns:
            dir_mask = (
                valid["last_known_price"].notna() &
                (valid["actual_price"] != valid["last_known_price"].astype(float))
            )
            dir_valid = valid[dir_mask]
            if not dir_valid.empty:
                actual_dir = np.sign(dir_valid["actual_price"] - dir_valid["last_known_price"].astype(float))
                pred_dir = np.sign(dir_valid["predicted_price"] - dir_valid["last_known_price"].astype(float))
                dir_acc = (actual_dir == pred_dir).mean()
                print(f"  Dir.Acc: {dir_acc:.2%} ({len(dir_valid):,} directional predictions)")
    else:
        print("No valid prediction pairs found (all null).")
else:
    print("No per-prediction data with actual/predicted columns.")

---
## 2. Category Breakdown

Aggregate metrics hide category-level variation. Consumables (stable, high-volume) are inherently easier to predict than collection items (volatile, low-volume). Breaking metrics down by category reveals where the model excels and where it struggles.

Categories with higher MAPE may still produce useful recommendations if the directional accuracy is good -- a model that correctly predicts "prices will rise" is useful even if it overestimates the magnitude.

In [ ]:
# ── MAE and MAPE by category ─────────────────────────────────────────────────

if not df_summary.empty:
    fig = plot_metrics_by_category(df_summary)
    plt.show()
else:
    # Fall back to computing from per-prediction data
    if not df_predictions.empty and "category_tag" in df_predictions.columns:
        valid = df_predictions[
            df_predictions["actual_price"].notna() &
            df_predictions["predicted_price"].notna()
        ].copy()
        valid["actual_price"] = valid["actual_price"].astype(float)
        valid["predicted_price"] = valid["predicted_price"].astype(float)
        valid["abs_error"] = (valid["actual_price"] - valid["predicted_price"]).abs()

        cat_metrics = valid.groupby("category_tag").agg(
            mae=("abs_error", "mean"),
            n=("abs_error", "count"),
        ).reset_index()

        if not cat_metrics.empty:
            fig, ax = plt.subplots(figsize=(12, 5))
            apply_wow_theme(fig)

            order = cat_metrics.sort_values("mae", ascending=False)
            colors = [CATEGORY_COLORS.get(c, WOW_PALETTE["primary"]) for c in order["category_tag"]]

            ax.barh(order["category_tag"], order["mae"], color=colors, alpha=0.8,
                    edgecolor=WOW_PALETTE["grid"], linewidth=0.5)
            ax.set_xlabel("MAE (gold)")
            ax.set_title("Mean Absolute Error by Category", fontweight="bold")
            fig.tight_layout()
            plt.show()
    else:
        print("No data available for category breakdown.")

In [ ]:
# ── Directional accuracy heatmap (category x horizon) ────────────────────────

if not df_predictions.empty and "direction_correct" in df_predictions.columns:
    fig = plot_directional_accuracy_heatmap(df_predictions)
    plt.show()
elif not df_predictions.empty and "last_known_price" in df_predictions.columns:
    # Compute direction_correct on the fly
    valid = df_predictions[
        df_predictions["actual_price"].notna() &
        df_predictions["predicted_price"].notna() &
        df_predictions["last_known_price"].notna()
    ].copy()
    valid["actual_price"] = valid["actual_price"].astype(float)
    valid["predicted_price"] = valid["predicted_price"].astype(float)
    valid["last_known_price"] = valid["last_known_price"].astype(float)

    # Exclude ties
    valid = valid[valid["actual_price"] != valid["last_known_price"]]

    if not valid.empty:
        valid["direction_correct"] = (
            np.sign(valid["actual_price"] - valid["last_known_price"]) ==
            np.sign(valid["predicted_price"] - valid["last_known_price"])
        ).astype(int)

        fig = plot_directional_accuracy_heatmap(valid)
        plt.show()
    else:
        print("Insufficient data for directional accuracy heatmap.")
else:
    print("No directional data available.")

---
## 3. Actual vs Predicted

The scatter plot of actual vs predicted prices is the classic ML evaluation chart. Points on the diagonal line represent perfect predictions. Systematic deviations indicate bias:

- Points **above** the line = underprediction (model is too conservative)
- Points **below** the line = overprediction (model is too optimistic)

The R-squared annotation quantifies the fraction of price variance explained by the model. R-squared > 0.8 is strong for financial forecasting.

In [ ]:
# ── Actual vs Predicted scatter ──────────────────────────────────────────────

if not df_predictions.empty and "actual_price" in df_predictions.columns:
    fig = plot_actual_vs_predicted_scatter(df_predictions, max_points=2000)
    plt.show()
else:
    print("No backtest predictions available for scatter plot.")

In [ ]:
# ── Residual distribution ────────────────────────────────────────────────────
# A well-calibrated model has residuals centered near zero with a symmetric distribution.
# Heavy tails indicate occasional large errors; skew indicates systematic bias.

if not df_predictions.empty and "actual_price" in df_predictions.columns:
    fig = plot_residual_distribution(df_predictions)
    plt.show()
else:
    print("No data for residual distribution.")

In [ ]:
# ── Walk-forward MAE stability ───────────────────────────────────────────────
# Ideally, MAE is stable or declining across folds (model improves with more data).
# A sharp MAE spike in a specific fold indicates an external shock (event, patch)
# that the model was not prepared for.

if not df_predictions.empty and "fold_index" in df_predictions.columns:
    fig = plot_walk_forward_mae(df_predictions)
    plt.show()
else:
    print("No fold-level data available for walk-forward MAE chart.")

---
## 4. Calibration Analysis

Point predictions tell us *where* we think the price will be. Confidence intervals tell us *how sure we are*. A well-calibrated model's X% CI should contain approximately X% of actual observations.

### CI methodology (current)

The current CI is **heuristic-based**, not model-derived:

1. Base width = `price_roll_std_7d * horizon_multiplier` (wider for longer horizons).
2. Drift adjustment: If drift detection flags anomalies, the uncertainty multiplier widens the CI.
3. Floor: CI lower bound cannot be less than 5% of current price.
4. Cap: CI upper bound cannot exceed 10x current price.
5. Cold-start: Additional 1.5x multiplier for archetypes with limited history.

### CI quality categories

- **good**: Width < 50% of predicted price -- model is confident.
- **wide**: Width 50-100% of predicted price -- usable but imprecise.
- **unreliable**: Width > 100% of predicted price -- forecast is noise.

Future improvement: Replace heuristic CIs with **quantile regression** (train separate LightGBM models at alpha=0.10 and alpha=0.90).

In [ ]:
# ── CI calibration check ─────────────────────────────────────────────────────
# For forecasts where we know the actual price, check what fraction of actuals
# fall within the CI.

df_forecasts = fetch_forecast_data(DB_PATH, REALM)

if not df_forecasts.empty and all(c in df_forecasts.columns for c in
        ["confidence_lower", "confidence_upper", "predicted_price_gold"]):

    # Try to match forecasts to actuals via the DB
    cal_sql = """
        SELECT fo.archetype_id, fo.forecast_horizon, fo.target_date,
               fo.predicted_price_gold, fo.confidence_lower, fo.confidence_upper,
               fo.ci_quality,
               (
                   SELECT AVG(n.price_gold)
                   FROM   market_observations_normalized n
                   JOIN   items i ON n.item_id = i.item_id
                   WHERE  i.archetype_id = fo.archetype_id
                     AND  n.realm_slug   = fo.realm_slug
                     AND  date(n.observed_at) = fo.target_date
                     AND  n.is_outlier = 0
               ) AS actual_price
        FROM   forecast_outputs fo
        WHERE  fo.realm_slug = ?
          AND  fo.archetype_id IS NOT NULL
    """
    try:
        conn = sqlite3.connect(str(DB_PATH))
        df_cal = pd.read_sql_query(cal_sql, conn, params=[REALM])
        conn.close()
    except Exception:
        df_cal = pd.DataFrame()

    if not df_cal.empty:
        df_cal = df_cal.dropna(subset=["actual_price"])

    if not df_cal.empty:
        df_cal["in_ci"] = (
            (df_cal["actual_price"].astype(float) >= df_cal["confidence_lower"].astype(float)) &
            (df_cal["actual_price"].astype(float) <= df_cal["confidence_upper"].astype(float))
        )

        overall_coverage = df_cal["in_ci"].mean()
        print(f"Overall CI coverage: {overall_coverage:.1%} of actuals fall within CI")
        print(f"(Based on {len(df_cal):,} forecast-actual pairs)\n")

        # Coverage by horizon
        if "forecast_horizon" in df_cal.columns:
            horizon_cov = df_cal.groupby("forecast_horizon")["in_ci"].agg(["mean", "count"])
            horizon_cov.columns = ["coverage", "n_pairs"]
            print("Coverage by horizon:")
            for h, row in horizon_cov.iterrows():
                print(f"  {h}: {row['coverage']:.1%} ({int(row['n_pairs'])} pairs)")

        # Coverage by CI quality
        if "ci_quality" in df_cal.columns:
            quality_cov = df_cal.groupby("ci_quality")["in_ci"].agg(["mean", "count"])
            quality_cov.columns = ["coverage", "n_pairs"]
            print("\nCoverage by CI quality:")
            for q, row in quality_cov.iterrows():
                print(f"  {q}: {row['coverage']:.1%} ({int(row['n_pairs'])} pairs)")
    else:
        print(
            "No forecast-actual pairs found for calibration.\n"
            "This requires forecasts whose target_date has already passed with actual price data."
        )
else:
    print("No forecast data with CI bounds available.")

In [ ]:
# ── CI coverage visualization ────────────────────────────────────────────────

if 'df_cal' in dir() and not df_cal.empty and "in_ci" in df_cal.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    apply_wow_theme(fig)

    # Left: coverage by horizon as bar chart
    ax1 = axes[0]
    if "forecast_horizon" in df_cal.columns:
        horizons = sorted(df_cal["forecast_horizon"].unique())
        coverages = [df_cal[df_cal["forecast_horizon"] == h]["in_ci"].mean() for h in horizons]
        colors = [HORIZON_COLORS.get(h, WOW_PALETTE["primary"]) for h in horizons]

        bars = ax1.bar(horizons, coverages, color=colors, alpha=0.8,
                       edgecolor=WOW_PALETTE["grid"], linewidth=0.5)
        ax1.axhline(0.80, color=WOW_PALETTE["accent_red"], linestyle="--",
                    alpha=0.6, linewidth=1.5, label="80% target")
        for bar, cov in zip(bars, coverages):
            ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                     f"{cov:.0%}", ha="center", va="bottom",
                     color=WOW_PALETTE["text"], fontsize=10, fontweight="bold")
        ax1.set_ylabel("Coverage (fraction in CI)")
        ax1.set_title("CI Coverage by Horizon", fontweight="bold")
        ax1.set_ylim(0, 1.1)
        ax1.legend(framealpha=0.8)

    # Right: CI width distribution
    ax2 = axes[1]
    df_cal["ci_width"] = (
        df_cal["confidence_upper"].astype(float) -
        df_cal["confidence_lower"].astype(float)
    )
    df_cal["ci_width_pct"] = df_cal["ci_width"] / df_cal["predicted_price_gold"].astype(float).clip(lower=0.01)

    in_ci = df_cal[df_cal["in_ci"]]["ci_width_pct"]
    out_ci = df_cal[~df_cal["in_ci"]]["ci_width_pct"]

    if not in_ci.empty:
        ax2.hist(in_ci.clip(upper=3), bins=30, alpha=0.6,
                 color="#2ECC71", label="Actual in CI",
                 edgecolor=WOW_PALETTE["grid"], linewidth=0.5)
    if not out_ci.empty:
        ax2.hist(out_ci.clip(upper=3), bins=30, alpha=0.6,
                 color="#E74C3C", label="Actual outside CI",
                 edgecolor=WOW_PALETTE["grid"], linewidth=0.5)
    ax2.set_xlabel("CI Width / Predicted Price")
    ax2.set_ylabel("Count")
    ax2.set_title("CI Width Distribution (Hit vs Miss)", fontweight="bold")
    ax2.legend(framealpha=0.8)

    fig.suptitle("Confidence Interval Calibration",
                 fontsize=14, fontweight="bold", color=WOW_PALETTE["text"])
    fig.tight_layout()
    plt.show()
else:
    print("No calibration data available for CI visualization.")

**Calibration insights:**

- If coverage is significantly above 80% (e.g., 95%), the CIs are too wide -- the model is being too cautious, reducing the usefulness of the intervals.
- If coverage is below 70%, the CIs are too narrow -- the model is overconfident.
- The "CI width for misses" distribution tells us whether the failures are near-misses (narrow CI, actual just outside) or catastrophic (narrow CI, actual far away).
- Moving to quantile regression CIs would improve calibration by learning the uncertainty directly from data.

---
## 5. Recommendation Trace

The recommendation engine converts forecasts into actionable signals using a weighted scoring formula:

```
score = 0.35 * opportunity   (ROI potential)
      + 0.20 * liquidity     (market depth)
      - 0.20 * volatility    (price instability risk)
      + 0.15 * event_boost   (event-driven demand)
      - 0.10 * uncertainty   (CI width / model confidence)
```

### Action determination

| Priority | Rule | Action |
|----------|------|--------|
| 1 | uncertainty_pct >= 95% | **AVOID** (forecast is noise) |
| 2 | ROI >= 10% | **BUY** |
| 3 | ROI <= -10% | **SELL** |
| 4 | All other | **HOLD** |

### Risk levels (independent of action)

| Level | Condition |
|-------|-----------|
| CRITICAL | uncertainty >= 95% |
| HIGH | uncertainty >= 80% OR volatility CV >= 80% |
| MEDIUM | uncertainty >= 50% OR volatility CV >= 50% |
| LOW | Everything else |

High-risk archetypes can still receive BUY/SELL actions -- the risk level is a separate dimension that surfaces uncertainty without suppressing profitable signals.

In [ ]:
# ── Load latest recommendations ──────────────────────────────────────────────

df_recs = fetch_recommendation_scores(RECS_DIR, REALM)

if not df_recs.empty:
    print(f"Recommendations loaded: {len(df_recs)}")
    print(f"\nAction distribution:")
    if "action" in df_recs.columns:
        print(df_recs["action"].value_counts().to_string())
    print(f"\nRisk level distribution:")
    if "risk_level" in df_recs.columns:
        print(df_recs["risk_level"].value_counts().to_string())
    print(f"\nCategories: {sorted(df_recs['category'].unique()) if 'category' in df_recs.columns else 'N/A'}")
else:
    print(
        "No recommendation data found.\n"
        "Run `wow-forecaster run-daily-forecast` to generate recommendations.\n"
        f"Expected location: {RECS_DIR}"
    )

In [ ]:
# ── Score tornado chart ──────────────────────────────────────────────────────
# Shows the weighted contribution of each score component for the top recommendations.

if not df_recs.empty and "score" in df_recs.columns:
    fig = plot_score_tornado(df_recs, top_n=10)
    plt.show()
else:
    print("No score data for tornado chart.")

In [ ]:
# ── Action and risk distribution ─────────────────────────────────────────────

if not df_recs.empty:
    fig = plot_action_distribution(df_recs)
    plt.show()
else:
    print("No recommendation data for distribution charts.")

In [ ]:
# ── ROI vs Uncertainty scatter ───────────────────────────────────────────────
# The decision landscape: high ROI + low uncertainty = best opportunities.

if not df_recs.empty:
    fig = plot_roi_vs_uncertainty(df_recs)
    plt.show()
else:
    print("No data for ROI vs Uncertainty chart.")

In [ ]:
# ── Detailed trace for a single recommendation ───────────────────────────────
# Walk through the score computation step-by-step for the top BUY recommendation.

if not df_recs.empty and "action" in df_recs.columns and "score" in df_recs.columns:
    buy_recs = df_recs[df_recs["action"] == "buy"].sort_values("score", ascending=False)
    if buy_recs.empty:
        buy_recs = df_recs.sort_values("score", ascending=False)  # fallback to any

    if not buy_recs.empty:
        top = buy_recs.iloc[0]
        print("=" * 60)
        print("RECOMMENDATION TRACE -- Top Scored Item")
        print("=" * 60)
        print(f"Category      : {top.get('category', 'N/A')}")
        print(f"Archetype ID  : {top.get('archetype_id', 'N/A')}")
        print(f"Horizon       : {top.get('horizon', 'N/A')}")
        print(f"Action        : {top.get('action', 'N/A')}")
        print(f"Risk Level    : {top.get('risk_level', 'N/A')}")
        print(f"Model         : {top.get('model_slug', 'N/A')}")
        print()
        print(f"Current Price : {top.get('current_price', 'N/A')}")
        print(f"Predicted     : {top.get('predicted_price', 'N/A')}")
        roi = top.get('roi_pct', None)
        if roi is not None:
            print(f"Expected ROI  : {float(roi) * 100:+.1f}%")
        if top.get('ci_lower') is not None and top.get('ci_upper') is not None:
            print(f"CI Range      : [{top['ci_lower']:.1f}, {top['ci_upper']:.1f}]")
        print()
        print("Score Components (weighted contribution):")
        print(f"  Opportunity  : {float(top.get('sc_opportunity', 0)):6.1f} x 0.35 = {float(top.get('sc_opportunity', 0)) * 0.35:+6.2f}")
        print(f"  Liquidity    : {float(top.get('sc_liquidity', 0)):6.1f} x 0.20 = {float(top.get('sc_liquidity', 0)) * 0.20:+6.2f}")
        print(f"  Volatility   : {float(top.get('sc_volatility', 0)):6.1f} x 0.20 = {float(top.get('sc_volatility', 0)) * -0.20:+6.2f}")
        print(f"  Event Boost  : {float(top.get('sc_event_boost', 0)):6.1f} x 0.15 = {float(top.get('sc_event_boost', 0)) * 0.15:+6.2f}")
        print(f"  Uncertainty  : {float(top.get('sc_uncertainty', 0)):6.1f} x 0.10 = {float(top.get('sc_uncertainty', 0)) * -0.10:+6.2f}")
        print(f"  ─────────────────────────────")
        print(f"  TOTAL SCORE  :                     {float(top.get('score', 0)):6.2f}")
else:
    print("No recommendation data for trace.")

In [ ]:
# ── Score component correlation ──────────────────────────────────────────────
# Are the components independent or redundant?

if not df_recs.empty:
    sc_cols = [c for c in df_recs.columns if c.startswith("sc_")]
    if sc_cols and len(sc_cols) >= 2:
        fig, ax = plt.subplots(figsize=(8, 6))
        apply_wow_theme(fig)

        sc_data = df_recs[sc_cols].astype(float)
        corr = sc_data.corr()

        # Rename columns for readability
        rename = {
            "sc_opportunity": "Opportunity",
            "sc_liquidity": "Liquidity",
            "sc_volatility": "Volatility",
            "sc_event_boost": "Event Boost",
            "sc_uncertainty": "Uncertainty",
        }
        corr = corr.rename(index=rename, columns=rename)

        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(
            corr, mask=mask, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            ax=ax, square=True,
            linewidths=0.5, linecolor=WOW_PALETTE["grid"],
            cbar_kws={"label": "Pearson Correlation"},
        )
        ax.set_title("Score Component Correlation", fontweight="bold")
        fig.tight_layout()
        plt.show()
    else:
        print("Insufficient score component columns for correlation analysis.")
else:
    print("No recommendation data for component analysis.")

---
## Summary

### Model strengths

- **Cross-archetype learning**: The global model shares patterns across categories, benefiting thin series.
- **Leakage-safe events**: The `announced_at` guard prevents the model from using future event knowledge.
- **Explainable scoring**: Every recommendation traces back to five interpretable components.
- **Cold-start handling**: Transfer blending + widened CIs provide honest uncertainty for new items.

### Known limitations

- **Heuristic CIs**: Current confidence intervals are rule-based, not learned. Quantile regression would improve calibration.
- **No hyperparameter tuning**: The model uses conservative defaults. Formal tuning (blocked on LightGBM backtest integration) could improve accuracy by 5-15%.
- **Market manipulation blind spot**: AH manipulation (e.g., price walls, buyout-and-relist) creates outliers that the z-score normalizer flags but cannot predict.
- **Single-region scope**: Each model trains on one region (US). Cross-region transfer is not implemented.

### Roadmap priorities

1. **LightGBM backtest integration** -- Evaluate the actual model in walk-forward fashion (current backtest covers baseline models only).
2. **Hyperparameter tuning** -- Optuna search over `min_child_samples`, `learning_rate`, `num_leaves`.
3. **Quantile CIs** -- Train alpha=0.10 and alpha=0.90 LightGBM models for calibrated intervals.
4. **Midnight data maturation** -- As Midnight accumulates 60+ days of data, retrain and re-evaluate.